# Deploy Streamlit to Azure VM

This notebook automates deploying the Streamlit frontend to your Azure VM.

**Prerequisites:**
- Azure VM from step `03-azure-vm-setup/` is running
- FastAPI is running on the VM (port 5000)
- Azure CLI authenticated (`az login`)

In [ ]:
!pip install azure-identity azure-mgmt-compute azure-mgmt-network python-dotenv

In [ ]:
import os
import subprocess
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.mgmt.compute import ComputeManagementClient
from azure.mgmt.network import NetworkManagementClient

load_dotenv()

SUBSCRIPTION_ID = os.environ['AZURE_SUBSCRIPTION_ID']
RESOURCE_GROUP = os.environ.get('AZURE_RESOURCE_GROUP', 'mlops-rg')
VM_NAME = os.environ.get('AZURE_VM_NAME', 'mlops-vm')
NSG_NAME = os.environ.get('AZURE_NSG_NAME', 'mlops-nsg')

credential = DefaultAzureCredential()
compute_client = ComputeManagementClient(credential, SUBSCRIPTION_ID)
network_client = NetworkManagementClient(credential, SUBSCRIPTION_ID)

print(f'Subscription:   {SUBSCRIPTION_ID}')
print(f'Resource Group: {RESOURCE_GROUP}')
print(f'VM:             {VM_NAME}')

In [ ]:
# Check VM status and get public IP
vm = compute_client.virtual_machines.get(
    RESOURCE_GROUP, VM_NAME, expand='instanceView'
)
statuses = vm.instance_view.statuses
power_state = next((s.display_status for s in statuses if s.code.startswith('PowerState')), 'Unknown')
print(f'VM status: {power_state}')

# Get public IP from NIC
nic_id = vm.network_profile.network_interfaces[0].id
nic_name = nic_id.split('/')[-1]
nic = network_client.network_interfaces.get(RESOURCE_GROUP, nic_name)
pip_id = nic.ip_configurations[0].public_ip_address.id
pip_name = pip_id.split('/')[-1]
public_ip = network_client.public_ip_addresses.get(RESOURCE_GROUP, pip_name)
EXTERNAL_IP = public_ip.ip_address
print(f'External IP: {EXTERNAL_IP}')

In [ ]:
# SCP Streamlit app to VM
streamlit_src = '../04-local-app-development/streamlit/app.py'
scp_cmd = ['scp', streamlit_src, f'azureuser@{EXTERNAL_IP}:~/streamlit-app/']

print('Running:', ' '.join(scp_cmd))
result = subprocess.run(scp_cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('Error:', result.stderr)

In [ ]:
# Create NSG rule to allow port 8501 (Streamlit)
from azure.mgmt.network.models import SecurityRule

rule_name = 'allow-streamlit'
nsg = network_client.network_security_groups.get(RESOURCE_GROUP, NSG_NAME)
existing_rules = [r.name for r in (nsg.security_rules or [])]

if rule_name in existing_rules:
    print(f'NSG rule {rule_name!r} already exists — skipping.')
else:
    rule = SecurityRule(
        protocol='Tcp',
        source_port_range='*',
        destination_port_range='8501',
        source_address_prefix='*',
        destination_address_prefix='*',
        access='Allow',
        priority=1100,
        direction='Inbound',
        description='Allow Streamlit traffic on port 8501',
    )
    op = network_client.security_rules.begin_create_or_update(
        RESOURCE_GROUP, NSG_NAME, rule_name, rule
    )
    op.result()
    print(f'NSG rule {rule_name!r} created.')

In [ ]:
# Install Streamlit and run in background via SSH
install_and_run = (
    'mkdir -p ~/streamlit-app && '
    'pip install --quiet streamlit requests && '
    'nohup streamlit run ~/streamlit-app/app.py '
    '--server.port 8501 --server.address 0.0.0.0 '
    '> ~/streamlit.log 2>&1 &'
)

ssh_cmd = ['ssh', f'azureuser@{EXTERNAL_IP}', install_and_run]
result = subprocess.run(ssh_cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('Error:', result.stderr)

In [ ]:
print(f'Streamlit is now running at:')
print(f'  http://{EXTERNAL_IP}:8501')